# Silver population place reconciliation

## Purpose

Create one explainable reconciliation result for every current Census Table C source row. Each row keeps its file, sheet, row number, raw place name, raw population, selected Bronze snapshot, and Bronze ingest run. Matching uses verified PSGC context first and active approved aliases second. Ambiguous, unmatched, and invalid rows remain visible. The natural key is source file, sheet, row number, and deterministic Silver run ID.


## 1. Set the version contract and safety gates

The source-supported population reference date is July 1, 2024. A single active alias version may participate. An empty alias table is valid and becomes `NO_APPROVED_ALIAS`. The latest Table C load must be successful and its latest blocking Bronze validation must pass. The PSGC Silver output must match the latest PSGC Bronze snapshot.


In [0]:
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW population_pipeline_parameters AS
SELECT
    DATE'2024-07-01' AS population_reference_date,
    2024 AS reference_year,
    'PSA 2024 Census Table C' AS source_name,
    'silver-population-place-v1' AS transformation_rule_version,
    COALESCE(MAX(alias_version), 'NO_APPROVED_ALIAS') AS alias_version,
    COUNT(DISTINCT alias_version) AS active_alias_version_count
FROM `02-silver`.config_place_name_alias
WHERE source_system = 'census_2024_table_c'
  AND approval_status = 'APPROVED'
  AND is_active = TRUE;

SELECT ASSERT_TRUE(
    active_alias_version_count <= 1,
    'Population matching requires at most one active approved Table C alias version'
)
FROM population_pipeline_parameters;

CREATE OR REPLACE TEMPORARY VIEW latest_population_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'census_2024_table_c'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'Table C Bronze safety gate failed: latest audit must be successful, identified, and non-empty'
)
FROM latest_population_load;

CREATE OR REPLACE TEMPORARY VIEW latest_population_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'census_2024_table_c'
      AND snapshot_id = (SELECT snapshot_id FROM latest_population_load)
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'Table C Bronze safety gate failed: latest blocking validation did not pass'
)
FROM latest_population_validation;

CREATE OR REPLACE TEMPORARY VIEW latest_psgc_load_for_population AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'psgc'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'PSGC Bronze safety gate failed before population matching'
)
FROM latest_psgc_load_for_population;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_snapshot_id) = 1
    AND MAX(source_snapshot_id) = (SELECT snapshot_id FROM latest_psgc_load_for_population),
    'silver_psgc_place is not aligned with the latest safe PSGC Bronze snapshot'
)
FROM `02-silver`.silver_psgc_place;


## 2. Preserve Table C rows and infer verified context

Table C provides file, sheet, row, place name, and population, but no reliable PSGC code. The pipeline safely parses population and retains its raw value. Sheet names establish province or independent-city context only when they resolve to one PSGC place. Uppercase locality headings establish the current city or municipality only when they resolve uniquely inside that sheet context. This context is carried to later barangay rows in source-row order.


In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_c_source AS
SELECT
    source_file,
    sheet_name,
    source_row_number,
    place_name AS place_name_raw,
    population AS population_raw,
    TRY_CAST(REPLACE(TRIM(population), ',', '') AS BIGINT) AS population_count,
    REGEXP_REPLACE(
        REGEXP_REPLACE(UPPER(TRIM(place_name)), '\\s+', ' '),
        '\\s*\\*+$',
        ''
    ) AS place_name_standardized,
    REGEXP_REPLACE(
        REGEXP_REPLACE(
            REGEXP_REPLACE(UPPER(TRIM(place_name)), '\\s+', ' '),
            '\\s*\\*+$',
            ''
        ),
        '\\s+[0-9]+$',
        ''
    ) AS heading_name_standardized,
    REGEXP_REPLACE(UPPER(TRIM(sheet_name)), '\\s+', ' ') AS sheet_name_standardized,
    REGEXP_REPLACE(TRIM(place_name), '\\s+', ' ') = UPPER(REGEXP_REPLACE(TRIM(place_name), '\\s+', ' '))
        AS is_uppercase_heading,
    MIN(TRY_CAST(source_row_number AS BIGINT)) OVER (PARTITION BY source_file, sheet_name)
        AS sheet_first_row_number,
    _source_snapshot_id AS source_snapshot_id,
    _ingest_run_id AS source_ingest_run_id,
    _ingested_at AS source_load_ts,
    SHA2(CONCAT_WS('|', _source_snapshot_id, source_file, sheet_name, source_row_number), 256)
        AS source_row_identity
FROM `01-bronze`.census_2024_table_c;

CREATE OR REPLACE TEMPORARY VIEW table_c_sheet_candidates AS
SELECT
    source.source_file,
    source.sheet_name,
    CASE
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) > 0
            THEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END)
        ELSE COUNT(DISTINCT CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END)
    END AS candidate_count,
    CASE
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) = 1
            THEN MAX(CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END)
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) = 0
         AND COUNT(DISTINCT CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END) = 1
            THEN MAX(CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END)
    END AS candidate_psgc_code
FROM (
    SELECT DISTINCT source_file, sheet_name, sheet_name_standardized
    FROM table_c_source
) AS source
LEFT JOIN `02-silver`.silver_psgc_place AS place
    ON source.sheet_name_standardized = place.place_name_standardized
   AND place.place_type IN ('PROVINCE', 'CITY', 'MUNICIPALITY')
GROUP BY source.source_file, source.sheet_name;

CREATE OR REPLACE TEMPORARY VIEW table_c_with_sheet_context AS
SELECT
    source.*,
    CASE WHEN sheet.candidate_count = 1 THEN sheet.candidate_psgc_code END AS sheet_psgc_code,
    sheet.candidate_count AS sheet_candidate_count,
    context.place_type AS sheet_place_type,
    context.region_code AS sheet_region_code,
    context.region_name AS sheet_region_name,
    context.province_code AS sheet_province_code,
    context.province_name AS sheet_province_name
FROM table_c_source AS source
LEFT JOIN table_c_sheet_candidates AS sheet
    ON source.source_file = sheet.source_file
   AND source.sheet_name = sheet.sheet_name
LEFT JOIN `02-silver`.silver_psgc_place AS context
    ON sheet.candidate_count = 1
   AND sheet.candidate_psgc_code = context.psgc_code;

CREATE OR REPLACE TEMPORARY VIEW table_c_locality_headings AS
SELECT
    source.source_row_identity,
    COUNT(DISTINCT place.psgc_code) AS candidate_count,
    MAX(place.psgc_code) AS candidate_psgc_code
FROM table_c_with_sheet_context AS source
LEFT JOIN `02-silver`.silver_psgc_place AS place
    ON (source.is_uppercase_heading OR place.place_type = 'SUBMUNICIPALITY')
   AND TRY_CAST(source.source_row_number AS BIGINT) <> source.sheet_first_row_number
   AND source.heading_name_standardized = place.place_name_standardized
   AND place.place_type IN ('CITY', 'MUNICIPALITY', 'SUBMUNICIPALITY')
   AND (
       (source.sheet_place_type = 'PROVINCE' AND place.province_code = source.sheet_psgc_code)
       OR (source.sheet_place_type IN ('CITY', 'MUNICIPALITY') AND place.psgc_code = source.sheet_psgc_code)
   )
GROUP BY source.source_row_identity;

CREATE OR REPLACE TEMPORARY VIEW table_c_with_locality_context AS
SELECT
    source.*,
    COALESCE(
        LAST_VALUE(
            CASE WHEN heading.candidate_count = 1 THEN heading.candidate_psgc_code END,
            TRUE
        ) OVER (
            PARTITION BY source.source_file, source.sheet_name
            ORDER BY TRY_CAST(source.source_row_number AS BIGINT)
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        CASE WHEN source.sheet_place_type IN ('CITY', 'MUNICIPALITY') THEN source.sheet_psgc_code END
    ) AS locality_context_code
FROM table_c_with_sheet_context AS source
LEFT JOIN table_c_locality_headings AS heading
    ON source.source_row_identity = heading.source_row_identity;


## 3. Apply the deterministic matching waterfall

Exact candidates must use verified sheet or locality context. A province or independent-city subtotal can match its unique sheet place. A locality heading can match its unique contextual PSGC locality. Other rows can match only a barangay under the verified current locality. Active `APPROVED` aliases may resolve a remaining row only for the selected alias version and with nonblank geographic context. Global name-only and fuzzy matching are not accepted methods.


In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_c_exact_candidates AS
WITH candidates AS (
    SELECT source.source_row_identity, place.psgc_code
    FROM table_c_with_locality_context AS source
    JOIN `02-silver`.silver_psgc_place AS place
        ON source.sheet_candidate_count = 1
       AND source.sheet_psgc_code = place.psgc_code
       AND TRY_CAST(source.source_row_number AS BIGINT) = source.sheet_first_row_number
       AND source.place_name_standardized = place.place_name_standardized
    UNION ALL
    SELECT source.source_row_identity, heading.candidate_psgc_code
    FROM table_c_with_locality_context AS source
    JOIN table_c_locality_headings AS heading
        ON source.source_row_identity = heading.source_row_identity
       AND heading.candidate_count = 1
    UNION ALL
    SELECT source.source_row_identity, place.psgc_code
    FROM table_c_with_locality_context AS source
    JOIN `02-silver`.silver_psgc_place AS place
        ON source.locality_context_code = place.city_municipality_code
       AND source.place_name_standardized = place.place_name_standardized
       AND NOT source.is_uppercase_heading
       AND place.place_type = 'BARANGAY'
)
SELECT
    source_row_identity,
    COUNT(DISTINCT psgc_code) AS candidate_count,
    MAX(psgc_code) AS candidate_psgc_code
FROM candidates
GROUP BY source_row_identity;

CREATE OR REPLACE TEMPORARY VIEW table_c_alias_candidates AS
SELECT
    source.source_row_identity,
    COUNT(DISTINCT target.psgc_code) AS candidate_count,
    MAX(target.psgc_code) AS candidate_psgc_code
FROM table_c_with_locality_context AS source
CROSS JOIN population_pipeline_parameters AS parameters
LEFT JOIN `02-silver`.silver_psgc_place AS locality
    ON source.locality_context_code = locality.psgc_code
JOIN `02-silver`.config_place_name_alias AS alias
    ON alias.source_system = 'census_2024_table_c'
   AND alias.alias_version = parameters.alias_version
   AND alias.approval_status = 'APPROVED'
   AND alias.is_active = TRUE
   AND REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ') = source.place_name_standardized
   AND (
       TRIM(alias.raw_region_name) <> ''
       OR TRIM(alias.raw_province_name) <> ''
       OR TRIM(alias.raw_city_municipality_name) <> ''
   )
   AND (TRIM(alias.raw_region_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_region_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_region_name)), '\\s+', ' '))
   AND (TRIM(alias.raw_province_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_province_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_province_name)), '\\s+', ' '))
   AND (TRIM(alias.raw_city_municipality_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_city_municipality_name)), '\\s+', ' ') = locality.place_name_standardized)
JOIN `02-silver`.silver_psgc_place AS target
    ON alias.canonical_psgc_code = target.psgc_code
   AND UPPER(alias.place_type) = target.place_type
   AND (alias.psgc_version IS NULL OR alias.psgc_version = target.psgc_version)
GROUP BY source.source_row_identity;


## 4. Publish one mutually exclusive result per source row

A unique exact contextual match wins first. Otherwise, one approved contextual alias may resolve the row. Multiple candidates remain ambiguous, and zero candidates remain unmatched. Population uses safe parsing and is never replaced by PSGC. The final disposition is mutually exclusive, so its counts must reconcile exactly to Table C rows. Only matched, positive barangay records become `ACCEPTED_PRIMARY` for regional aggregation.


In [0]:
CREATE OR REPLACE TABLE `02-silver`.silver_population_place_reconciliation
USING DELTA
COMMENT 'One explainable PSGC reconciliation result per current Table C source row'
AS
WITH resolved AS (
    SELECT
        source.*,
        COALESCE(exact.candidate_count, 0) AS exact_candidate_count,
        COALESCE(alias.candidate_count, 0) AS alias_candidate_count,
        CASE
            WHEN COALESCE(exact.candidate_count, 0) = 1 THEN exact.candidate_psgc_code
            WHEN COALESCE(alias.candidate_count, 0) = 1 THEN alias.candidate_psgc_code
        END AS matched_psgc_code,
        CASE
            WHEN source.source_file IS NULL OR source.sheet_name IS NULL
              OR source.source_row_number IS NULL OR NULLIF(TRIM(source.place_name_raw), '') IS NULL
                THEN 'INVALID_SOURCE'
            WHEN COALESCE(exact.candidate_count, 0) = 1 THEN 'MATCHED_EXACT_CONTEXT'
            WHEN COALESCE(alias.candidate_count, 0) = 1 THEN 'MATCHED_ALIAS'
            WHEN COALESCE(exact.candidate_count, 0) > 1 OR COALESCE(alias.candidate_count, 0) > 1
                THEN 'AMBIGUOUS'
            ELSE 'UNMATCHED'
        END AS match_status,
        parameters.* EXCEPT (active_alias_version_count)
    FROM table_c_with_locality_context AS source
    CROSS JOIN population_pipeline_parameters AS parameters
    LEFT JOIN table_c_exact_candidates AS exact
        ON source.source_row_identity = exact.source_row_identity
    LEFT JOIN table_c_alias_candidates AS alias
        ON source.source_row_identity = alias.source_row_identity
),
enriched AS (
    SELECT
        resolved.*,
        target.place_name_raw AS matched_place_name,
        target.place_type AS matched_place_type,
        target.region_code AS matched_region_code,
        target.region_name AS matched_region_name,
        target.province_code AS matched_province_code,
        target.city_municipality_code AS matched_city_municipality_code,
        target.psgc_population_crosscheck,
        target.psgc_version,
        target.source_snapshot_id AS psgc_source_snapshot_id,
        target.source_ingest_run_id AS psgc_source_ingest_run_id
    FROM resolved
    LEFT JOIN `02-silver`.silver_psgc_place AS target
        ON resolved.matched_psgc_code = target.psgc_code
)
SELECT
    source_file,
    sheet_name,
    source_row_number,
    source_row_identity,
    place_name_raw,
    place_name_standardized,
    population_raw,
    population_count,
    matched_psgc_code,
    matched_place_name,
    matched_place_type,
    matched_region_code,
    matched_region_name,
    matched_province_code,
    matched_city_municipality_code,
    match_status,
    CASE match_status
        WHEN 'MATCHED_EXACT_CONTEXT' THEN 'EXACT_CONTEXT'
        WHEN 'MATCHED_ALIAS' THEN 'APPROVED_ALIAS'
        ELSE 'NO_ACCEPTED_METHOD'
    END AS match_method,
    CASE
        WHEN match_status = 'MATCHED_EXACT_CONTEXT' THEN 'HIGH'
        WHEN match_status = 'MATCHED_ALIAS' THEN 'REVIEWED'
        ELSE 'UNRESOLVED'
    END AS match_quality,
    GREATEST(exact_candidate_count, alias_candidate_count) AS candidate_match_count,
    CASE
        WHEN match_status = 'AMBIGUOUS' THEN CONCAT('Multiple contextual candidates: exact=', exact_candidate_count, ', alias=', alias_candidate_count)
        WHEN match_status = 'UNMATCHED' AND sheet_candidate_count <> 1 THEN CONCAT('Sheet context candidate count=', sheet_candidate_count)
        WHEN match_status = 'UNMATCHED' AND locality_context_code IS NULL THEN 'No verified locality context'
        WHEN match_status = 'INVALID_SOURCE' THEN 'Missing mandatory Table C source identity or place name'
    END AS ambiguity_or_exception_reason,
    CASE
        WHEN population_count IS NULL OR population_count <= 0 THEN 'INVALID_POPULATION'
        WHEN match_status = 'INVALID_SOURCE' THEN 'SOURCE_EXCEPTION'
        WHEN match_status = 'AMBIGUOUS' THEN 'AMBIGUOUS'
        WHEN match_status = 'UNMATCHED' THEN 'UNMATCHED'
        WHEN matched_place_type = 'BARANGAY' THEN 'ACCEPTED_PRIMARY'
        ELSE 'MATCHED_NON_PRIMARY_LEVEL'
    END AS row_disposition,
    psgc_population_crosscheck,
    CASE
        WHEN population_count IS NOT NULL AND psgc_population_crosscheck IS NOT NULL
            THEN population_count - psgc_population_crosscheck
    END AS population_difference,
    CASE
        WHEN population_count > 0 AND psgc_population_crosscheck IS NOT NULL
            THEN ROUND(100.0 * (population_count - psgc_population_crosscheck) / population_count, 4)
    END AS population_difference_pct,
    population_reference_date,
    reference_year,
    source_name,
    source_snapshot_id,
    source_ingest_run_id,
    psgc_source_snapshot_id,
    psgc_source_ingest_run_id,
    psgc_version,
    alias_version,
    transformation_rule_version,
    SHA2(CONCAT_WS('|', source_snapshot_id, psgc_source_snapshot_id, alias_version, transformation_rule_version), 256) AS run_id,
    source_load_ts
FROM enriched;


# Summary

Created `02-silver.silver_population_place_reconciliation` at one result per Table C source row and Silver run. Its natural key retains source file, sheet, row number, and run ID. The table records the Table C and PSGC snapshots, both Bronze ingest identities, PSGC and alias versions, match method, final row disposition, population parsing, and diagnostic cross-check. Every Table C row stays visible. Requires Databricks execution. Next dependency: region population aggregation.


In [0]:
SELECT
    source_snapshot_id,
    psgc_source_snapshot_id,
    alias_version,
    match_status,
    row_disposition,
    COUNT(*) AS source_rows
FROM `02-silver`.silver_population_place_reconciliation
GROUP BY source_snapshot_id, psgc_source_snapshot_id, alias_version, match_status, row_disposition
ORDER BY row_disposition, match_status;
